In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder


In [17]:
## Load the dataset
data = pd.read_csv('./training_data/Churn_Modelling.csv')
print(data.head())

   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0  
4         790

In [18]:
## Preprocess the data
### Drop irrelevant columns
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)
print(data.head())

   CreditScore Geography  Gender  Age  Tenure    Balance  NumOfProducts  \
0          619    France  Female   42       2       0.00              1   
1          608     Spain  Female   41       1   83807.86              1   
2          502    France  Female   42       8  159660.80              3   
3          699    France  Female   39       1       0.00              2   
4          850     Spain  Female   43       2  125510.82              1   

   HasCrCard  IsActiveMember  EstimatedSalary  Exited  
0          1               1        101348.88       1  
1          0               1        112542.58       0  
2          1               0        113931.57       1  
3          0               0         93826.63       0  
4          1               1         79084.10       0  


In [19]:
## Encode categorical variables
label_encoder_gender = LabelEncoder()
data["Gender"] = np.asarray(label_encoder_gender.fit_transform(data['Gender']))
print(data.head())

   CreditScore Geography  Gender  Age  Tenure    Balance  NumOfProducts  \
0          619    France       0   42       2       0.00              1   
1          608     Spain       0   41       1   83807.86              1   
2          502    France       0   42       8  159660.80              3   
3          699    France       0   39       1       0.00              2   
4          850     Spain       0   43       2  125510.82              1   

   HasCrCard  IsActiveMember  EstimatedSalary  Exited  
0          1               1        101348.88       1  
1          0               1        112542.58       0  
2          1               0        113931.57       1  
3          0               0         93826.63       0  
4          1               1         79084.10       0  


In [20]:
## Onehot encode 'Geography'
## Scope of improvement can we use word_2_vec vector encoding ?

from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo = OneHotEncoder(sparse_output=False)
geo_encoder = onehot_encoder_geo.fit_transform(data[['Geography']])

print(geo_encoder)

[[1. 0. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 ...
 [1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]]


In [21]:
geo_feature_names = onehot_encoder_geo.get_feature_names_out(['Geography'])
print(geo_feature_names)

['Geography_France' 'Geography_Germany' 'Geography_Spain']


In [22]:
geo_encoder_df = pd.DataFrame(geo_encoder,columns=geo_feature_names)
print(geo_encoder_df)

      Geography_France  Geography_Germany  Geography_Spain
0                  1.0                0.0              0.0
1                  0.0                0.0              1.0
2                  1.0                0.0              0.0
3                  1.0                0.0              0.0
4                  0.0                0.0              1.0
...                ...                ...              ...
9995               1.0                0.0              0.0
9996               1.0                0.0              0.0
9997               1.0                0.0              0.0
9998               0.0                1.0              0.0
9999               1.0                0.0              0.0

[10000 rows x 3 columns]


In [23]:
## Combine one hot encoder columns with the original data
data = pd.concat([data.drop(['Geography'],axis=1),geo_encoder_df],axis=1)
print(data.head())

   CreditScore  Gender  Age  Tenure    Balance  NumOfProducts  HasCrCard  \
0          619       0   42       2       0.00              1          1   
1          608       0   41       1   83807.86              1          0   
2          502       0   42       8  159660.80              3          1   
3          699       0   39       1       0.00              2          0   
4          850       0   43       2  125510.82              1          1   

   IsActiveMember  EstimatedSalary  Exited  Geography_France  \
0               1        101348.88       1               1.0   
1               1        112542.58       0               0.0   
2               0        113931.57       1               1.0   
3               0         93826.63       0               1.0   
4               1         79084.10       0               0.0   

   Geography_Germany  Geography_Spain  
0                0.0              0.0  
1                0.0              1.0  
2                0.0              0.0 

In [24]:
## Save the encoders and scaler
import pickle


with open('./pkl/label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file=file)

with open('./pkl/onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file=file)

In [25]:
## DiVide the dataset into indepent and dependent features

X = data.drop(['Exited'],axis=1)
Y = data['Exited']

## Split the data in training and tetsing sets
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

## Scale these features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(X_train)

[[ 0.35649971  0.91324755 -0.6557859  ...  1.00150113 -0.57946723
  -0.57638802]
 [-0.20389777  0.91324755  0.29493847 ... -0.99850112  1.72572313
  -0.57638802]
 [-0.96147213  0.91324755 -1.41636539 ... -0.99850112 -0.57946723
   1.73494238]
 ...
 [ 0.86500853 -1.09499335 -0.08535128 ...  1.00150113 -0.57946723
  -0.57638802]
 [ 0.15932282  0.91324755  0.3900109  ...  1.00150113 -0.57946723
  -0.57638802]
 [ 0.47065475  0.91324755  1.15059039 ... -0.99850112  1.72572313
  -0.57638802]]


In [26]:
with open('./pkl/scaler.pkl','wb') as file:
    pickle.dump(scaler,file=file)

##### ### ANN Implementation

In [27]:
## Build Our ANN Model

from keras import Sequential , Input
from keras.layers import Dense
from keras.optimizers import Adam
from keras.losses import BinaryCrossentropy
import datetime

model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64,activation="relu"),
    Dense(32, activation='relu'),       # HL2
    Dense(1, activation='sigmoid')
])

print(model.summary())


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

None


In [28]:
## compile the model

opt = Adam(learning_rate=0.01)
loss = BinaryCrossentropy()
model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])

In [29]:
## Set up the Tensorboard
from keras.callbacks import EarlyStopping , TensorBoard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

## Set up Early Stopping
early_stopping_callback = EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)

In [30]:
### Train the model
history = model.fit(X_train,Y_train,validation_data=(X_test,Y_test),epochs=100,callbacks=[tensorflow_callback,early_stopping_callback])

print(history)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 802us/step - accuracy: 0.8339 - loss: 0.3949 - val_accuracy: 0.8420 - val_loss: 0.3760
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 494us/step - accuracy: 0.8533 - loss: 0.3565 - val_accuracy: 0.8460 - val_loss: 0.3751
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 467us/step - accuracy: 0.8583 - loss: 0.3502 - val_accuracy: 0.8625 - val_loss: 0.3457
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 473us/step - accuracy: 0.8581 - loss: 0.3458 - val_accuracy: 0.8570 - val_loss: 0.3417
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 465us/step - accuracy: 0.8626 - loss: 0.3393 - val_accuracy: 0.8610 - val_loss: 0.3363
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 479us/step - accuracy: 0.8616 - loss: 0.3370 - val_accuracy: 0.8500 - val_loss: 0.3449
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 466us/step - accuracy: 0.8615 - loss: 0.3370 - val_accuracy: 0.8610 - val_loss: 0.3407
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 470us/step - accuracy: 0.8625 - loss: 0

In [32]:
# save the modal

model.save('./model/model.keras')

In [33]:
## Load Tensorboard Extension
%load_ext tensorboard

In [35]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 5539), started 0:01:21 ago. (Use '!kill 5539' to kill it.)